# SBC Cross-Simulation

Detailed workflow for this experiment. For catalogue inputs and dependencies, see [data](../../../docs/data.ipynb). Enable `LG_RUN_RESEARCH_WORKFLOW=1` before running the calculation cells.

In [ ]:
import os
RUN_RESEARCH_WORKFLOW = os.environ.get("LG_RUN_RESEARCH_WORKFLOW") == "1"
if not RUN_RESEARCH_WORKFLOW:
    raise RuntimeError("Optional workflow disabled. Read notebooks/92_reproduce_saved_analyses.ipynb and set LG_RUN_RESEARCH_WORKFLOW=1.")

In [ ]:
from pathlib import Path
import os
ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "pyproject.toml").is_file())
os.environ.setdefault("LG_PROJECT", str(ROOT / "research"))
PROJECT = Path(os.environ["LG_PROJECT"]).expanduser().resolve()
os.chdir(PROJECT)

def design_path(logical_name):
    if logical_name == "FINAL_EXPERIMENT_DESIGN_2026-10-01.md":
        return PROJECT / "out/final_stage_audit/portability_v3/original_design.txt"
    return PROJECT / logical_name

### Corrected catalogue selection
Corrected TNG selection and results use separate v3 paths. Build the sample with
`python -X utf8 -B TNG300/rebuild_comparison.py`, then run `inference_npe.ipynb`
before dependent analyses in the original Roihu training environment.
The main FLAMINGO models are reused; their LG samples are refreshed after converting
physical observed radial velocity to peculiar velocity. The scalar affine MAF is
conditional Gaussian in log10 mass before posterior support restriction. Marginalizing
uncertain features can produce a non-Gaussian mixture. Calibration reports actual
central-interval coverage and descriptive discrete ranks; beta=0.6745 is only a
Gaussian reference. Historical source and outputs are preserved in
`review_backups/20260906_210956`; the working outputs must be regenerated.


In [ ]:
import os, sys
from pathlib import Path
PROJECT = Path(os.environ.get("LG_PROJECT", r"<research-project>"))
sys.path.insert(0, str(PROJECT / "analogues"))
from review_fixes import (RUN_VERSION, MODEL_DESCRIPTION, H0_BY_SIM, configure_run,
    feature_cache_dir, lg_observation, lg_to_peculiar, draw_noisy_feature,
    sample_lg_point, sample_lg_marginalized, calibration_summary, tng_volume_mpc3)
LEGACY_RESULTS_DIR = PROJECT / "pickles_v2"
FEATURE_CACHE_DIR = feature_cache_dir(PROJECT)

from review_fixes import require_legacy_environment
require_legacy_environment(PROJECT)


In [ ]:
import os, sys
from pathlib import Path
PROJECT  = Path(os.environ.get('LG_PROJECT', r'<research-project>'))
TNG_BASE = os.environ.get('TNG_BASE', r'<TNG_BASE>')
sys.path.insert(0, str(PROJECT / 'analogues'))

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pickle
import functools
from pathlib import Path
import torch
from scipy.stats import uniform, kstest
from sklearn.model_selection import train_test_split
import sys
sys.path.insert(0, str(PROJECT / "analogues"))
from splits import pair_groups, group_split

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

In [ ]:
results_dir = configure_run(PROJECT)      
cache_dir   = PROJECT / "analogues/cache/run_001_mstar2e10"
tng_dir     = PROJECT / "analogues/tng_for_comparison_v3"
results_dir.mkdir(parents=True, exist_ok=True)

def load_checkpoint_any_device(path, device):
    original_torch_load = torch.load
    torch.load = functools.partial(original_torch_load, map_location=device)
    try:
        with open(path, "rb") as f:
            obj = pickle.load(f)
    finally:
        torch.load = original_torch_load
    return obj

flam_ckpt = load_checkpoint_any_device(results_dir / "flamingo_npe_checkpoint.pkl", device)
flam_posteriors = flam_ckpt["posteriors"]
flam_scalers    = flam_ckpt["scalers"]

BLOCKS = {
    "core_dynamics":   ["separation", "vr", "vt", "log_mass_ratio"],
    "core_chemistry":  ["separation", "vr", "vt", "log_mass_ratio", "met_diff"],
    "core_morphology": ["separation", "vr", "vt", "log_mass_ratio",
                        "sigma_mean", "sigma_diff", "kappa_mean", "kappa_diff"],
    "core_photometry": ["separation", "vr", "vt", "log_mass_ratio", "gr_mean", "gr_diff"],
    "all_features":    ["separation", "vr", "vt", "log_mass_ratio",
                        "sigma_mean", "sigma_diff", "kappa_mean", "kappa_diff",
                        "met_diff", "gr_mean", "gr_diff"],
}
TARGET = "log_m200c"

# TNG raw data
with open(FEATURE_CACHE_DIR / "tng_sigma_star.pkl", "rb") as f:
    tng_sigma_star = pickle.load(f)
with open(FEATURE_CACHE_DIR / "tng_kappa_corot.pkl", "rb") as f:
    kappa_cache = pickle.load(f)
with open(FEATURE_CACHE_DIR / "metallicity_log_zscore.pkl", "rb") as f:
    met_transform = pickle.load(f)

keep_idx    = np.load(tng_dir / "keep_idx.npy")
tng_pairs_i = np.load(tng_dir / "pairs_i.npy")
tng_pairs_j = np.load(tng_dir / "pairs_j.npy")
tng_gr_sample   = np.load(tng_dir / "g_r.npy")
tng_met_sample  = np.load(tng_dir / "metallicity.npy")
tng_separation  = np.load(tng_dir / "separation.npy")
tng_vr          = np.load(tng_dir / "vr.npy")
tng_vt          = np.load(tng_dir / "vt.npy")
tng_log_m200c   = np.load(tng_dir / "log_m200c.npy")
tng_log_mass_ratio = np.load(tng_dir / "log_mass_ratio.npy")
tng_grnr = np.load(tng_dir / "grnr.npy")
tng_same_host = np.load(tng_dir / "have_same_host.npy").astype(bool)

tng_pairs_i_global = keep_idx[tng_pairs_i]
tng_pairs_j_global = keep_idx[tng_pairs_j]
tng_sigma_i = np.array([tng_sigma_star.get(int(x), np.nan) for x in tng_pairs_i_global])
tng_sigma_j = np.array([tng_sigma_star.get(int(x), np.nan) for x in tng_pairs_j_global])
tng_kappa_i = np.array([kappa_cache.get(int(x), np.nan) for x in tng_pairs_i_global])
tng_kappa_j = np.array([kappa_cache.get(int(x), np.nan) for x in tng_pairs_j_global])

def pair_features_tng(tng_pairs_i, tng_pairs_j, tng_gr_sample, tng_met_sample,
                       tng_sigma_i, tng_sigma_j, tng_kappa_i, tng_kappa_j,
                       tng_separation, tng_vr, tng_vt, tng_log_mass_ratio,
                       tng_log_m200c, met_transform):
    gr_i, gr_j = tng_gr_sample[tng_pairs_i], tng_gr_sample[tng_pairs_j]
    met_raw_i, met_raw_j = tng_met_sample[tng_pairs_i], tng_met_sample[tng_pairs_j]
    met_i = met_transform["tng_met_dex"][tng_pairs_i]
    met_j = met_transform["tng_met_dex"][tng_pairs_j]
    return pd.DataFrame({
        "gr_mean": (gr_i+gr_j)/2, "gr_diff": np.abs(gr_i-gr_j),
        "met_mean": (met_i+met_j)/2, "met_diff": np.abs(met_i-met_j),
        "sigma_mean": (tng_sigma_i+tng_sigma_j)/2, "sigma_diff": np.abs(tng_sigma_i-tng_sigma_j),
        "kappa_mean": (tng_kappa_i+tng_kappa_j)/2, "kappa_diff": np.abs(tng_kappa_i-tng_kappa_j),
        "separation": tng_separation, "vr": tng_vr, "vt": tng_vt,
        "log_mass_ratio": tng_log_mass_ratio, "log_m200c": tng_log_m200c,
        "host_i": tng_grnr[tng_pairs_i],
        "host_j": tng_grnr[tng_pairs_j],
    })

tng_df = pair_features_tng(
    tng_pairs_i, tng_pairs_j, tng_gr_sample, tng_met_sample,
    tng_sigma_i, tng_sigma_j, tng_kappa_i, tng_kappa_j,
    tng_separation, tng_vr, tng_vt, tng_log_mass_ratio,
    tng_log_m200c, met_transform
)
tng_df["same_host"] = tng_same_host

ALL_FEATURES = BLOCKS["all_features"]
tng_df_clean = tng_df.dropna(subset=ALL_FEATURES + [TARGET])

def get_test_split(df, feats, seed=42, test_size=0.15):
    cols  = feats + [TARGET, "host_i", "host_j"]
    valid = df[cols].dropna(subset=feats + [TARGET]).reset_index(drop=True)
    _, te = group_split(valid, test_size=test_size, seed=seed)
    return valid.iloc[te]

def tng_eval_set(df, feats, split_name):
    """Whole matched TNG population, no train/test split.

    A FLAMINGO-trained model has never seen any TNG data, so the entire TNG
    sample is out-of-sample; splitting it discards ~85% of the evaluation set
    for no reason. Matching the TNG population to the split the model was
    trained on also removes the same-host/diff-host mismatch: TNG has a different same-host fraction while a diff_host model saw none.
    """
    if split_name == "full":
        sub = df
    elif split_name == "diff_host":
        sub = df[~df["same_host"]]
    elif split_name == "same_host":
        sub = df[df["same_host"]]
    else:
        raise ValueError(f"unknown split: {split_name}")
    cols = feats + [TARGET, "host_i", "host_j"]
    return sub[cols].dropna(subset=feats + [TARGET]).reset_index(drop=True)

print(f"TNG evaluation sets — full: {len(tng_df_clean):,}  "
      f"same-host: {tng_df_clean['same_host'].sum():,}  "
      f"diff-host: {(~tng_df_clean['same_host']).sum():,}")

print(f"Loaded {len(flam_posteriors)} FLAMINGO combos. TNG: {len(tng_df_clean):,}")

In [ ]:
def evaluate_calibration(posterior, scaler, feats, test_df, n_posterior_samples=200):
    X = scaler.transform(test_df[feats].values.astype(np.float32))
    theta_true = test_df["log_m200c"].values.astype(np.float32)
    with torch.no_grad():
        samples = posterior.sample_batched((n_posterior_samples,),
            x=torch.tensor(X, device=device)).squeeze(-1).cpu().numpy()
    result = calibration_summary(samples, theta_true)
    result["evaluation_rows"] = test_df.index.to_numpy()
    for column in ("host_i", "host_j"):
        if column in test_df:
            result[column] = test_df[column].to_numpy()
    return result

cross_sim_sbc_results = {}
for split_name in ["full", "same_host", "diff_host"]:
    for block_name, feats in BLOCKS.items():
        key = f"{block_name}__{split_name}"
        tng_test = tng_eval_set(tng_df_clean, feats, split_name)
        cross_sim_sbc_results[key] = evaluate_calibration(
            flam_posteriors[key], flam_scalers[key], feats, tng_test)
with open(results_dir / "cross_sim_sbc_results.pkl", "wb") as f:
    pickle.dump(cross_sim_sbc_results, f)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, split_name in zip(axes, ["full", "same_host", "diff_host"]):
    for block in BLOCKS:
        r = cross_sim_sbc_results[f"{block}__{split_name}"]
        ax.plot(r["coverage_levels"], r["coverage"], "o-", label=block)
    ax.plot([0, 1], [0, 1], "k--")
    ax.set(xlabel="nominal central interval", ylabel="empirical coverage", title=split_name)
    ax.legend(fontsize=7)
plt.tight_layout()
plt.savefig(results_dir / "cross_sim_calibration.pdf", bbox_inches="tight")
plt.show()

In [ ]:
# FLAMINGO-only ablation model remains usable; evaluate it on corrected TNG rows.
ablation_ckpt = load_checkpoint_any_device(LEGACY_RESULTS_DIR / "ablation_reference_checkpoint.pkl", device)
no_sigma_feats = ["separation", "vr", "vt", "log_mass_ratio", "kappa_mean", "kappa_diff"]
posterior = ablation_ckpt["posteriors"]["morphology_no_sigma__flam"]
scaler = ablation_ckpt["scalers"]["morphology_no_sigma__flam"]
tng_test = tng_eval_set(tng_df_clean, no_sigma_feats, "full")
cross_sim_sbc_results["morphology_no_sigma__full"] = evaluate_calibration(
    posterior, scaler, no_sigma_feats, tng_test)
with open(results_dir / "cross_sim_sbc_results.pkl", "wb") as f:
    pickle.dump(cross_sim_sbc_results, f)
summary = pd.DataFrame([{ "key": key, **{name: r[name] for name in
    ("median_beta", "rank_cdf_distance", "coverage68", "coverage95", "bias", "n_test")}}
    for key, r in cross_sim_sbc_results.items()])
print(summary.to_string(index=False))
summary.to_csv(results_dir / "cross_sim_sbc_summary.csv", index=False)

In [ ]:
from sklearn.metrics import r2_score

def posterior_point_predictions(posterior, X_raw, scaler, device, n_samples=200):
    X_sc = scaler.transform(X_raw.astype(np.float32))
    x_batch = torch.tensor(X_sc, device=device)
    with torch.no_grad():
        samples = posterior.sample_batched((n_samples,), x=x_batch)
    return samples.mean(dim=0).squeeze(-1).cpu().numpy()

cross_sim_r2_matched = {}
print(f"{'model':<30} {'N(TNG)':>8} {'cross-sim R2':>14}")
print("-" * 56)
for split_name in ["full", "same_host", "diff_host"]:
    for block_name, feats in BLOCKS.items():
        key = f"{block_name}__{split_name}"
        ev  = tng_eval_set(tng_df_clean, feats, split_name)
        preds = posterior_point_predictions(flam_posteriors[key], ev[feats].values,
                                            flam_scalers[key], device)
        r2 = r2_score(ev[TARGET].values, preds)
        cross_sim_r2_matched[key] = {"r2": float(r2), "n": int(len(ev))}
        print(f"{key:<30} {len(ev):>8,} {r2:>14.3f}")

with open(results_dir / "cross_sim_r2_matched.pkl", "wb") as f:
    pickle.dump(cross_sim_r2_matched, f)

In [ ]:
from sklearn.metrics import mean_squared_error

print(f"{'model':<30} {'N':>6} {'target std':>11} {'RMSE':>8} {'bias':>8} {'R2':>8}")
for split_name in ["full", "same_host", "diff_host"]:
    for block_name, feats in BLOCKS.items():
        key = f"{block_name}__{split_name}"
        ev  = tng_eval_set(tng_df_clean, feats, split_name)
        y   = ev[TARGET].values
        preds = posterior_point_predictions(flam_posteriors[key], ev[feats].values,
                                            flam_scalers[key], device)
        rmse = np.sqrt(mean_squared_error(y, preds))
        print(f"{key:<30} {len(y):>6,} {y.std():>11.3f} {rmse:>8.3f} "
              f"{(preds - y).mean():>+8.3f} {r2_score(y, preds):>8.3f}")